<a href="https://colab.research.google.com/github/jairaj023/Internship-Practice/blob/main/Day_12Task.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install spacy

In [ ]:
import spacy
import pandas as pd
from spacy.training import Example
from spacy.matcher import PhraseMatcher

In [ ]:
from google.colab import files
upload = files.upload()

Saving raw_jobs.csv to raw_jobs.csv


In [ ]:
df = pd.read_csv("raw_jobs.csv")

In [ ]:
print(df.head())

     job_id         job_title  company              location  \
0  JOB00001  Python Developer  Infosys                Remote   
1  JOB00002    Data Scientist  Infosys  Hyderabad, Telangana   
2  JOB00003     Data Engineer      IBM  Bengaluru, Karnataka   
3  JOB00004  Business Analyst   Google  Hyderabad, Telangana   
4  JOB00005  Business Analyst  Mphasis          Delhi, India   

                                     job_description  experience education  \
0  We are looking for a Python Developer to join ...   1-3 years       MCA   
1  We are looking for a Data Scientist to join ou...   0-2 years       MCA   
2  We are looking for a Data Engineer to join our...  8-12 years       BCA   
3  We are looking for a Business Analyst to join ...   1-3 years    M.Tech   
4  We are looking for a Business Analyst to join ...  8-12 years    M.Tech   

       salary    job_type  
0  ₹10-16 LPA   Part-time  
1  ₹12-20 LPA      Remote  
2    ₹3-5 LPA  Internship  
3  ₹10-16 LPA      Remote  
4    ₹

In [ ]:
print(df.columns)

Index(['job_id', 'job_title', 'company', 'location', 'job_description',
       'experience', 'education', 'salary', 'job_type'],
      dtype='object')


In [ ]:
text_column = "job_description"

In [ ]:
print(df[text_column].head())

0    We are looking for a Python Developer to join ...
1    We are looking for a Data Scientist to join ou...
2    We are looking for a Data Engineer to join our...
3    We are looking for a Business Analyst to join ...
4    We are looking for a Business Analyst to join ...
Name: job_description, dtype: object


In [ ]:
nlp = spacy.load("en_core_web_sm")

In [ ]:
text = str(df[text_column].iloc[0])

doc = nlp(text)

for ent in doc.ents:
    print(ent.text, "->", ent.label_)

Infosys -> ORG
SQL -> ORG
Flask -> GPE
1-3 years -> DATE
MCA -> ORG


In [ ]:
skill_dictionary = {

    "SKILL": [
        "Python",
        "Java",
        "C++",
        "SQL",
        "Machine Learning",
        "Deep Learning",
        "Data Science",
        "Natural Language Processing"
    ],

    "TOOL": [
        "Power BI",
        "Tableau",
        "Excel",
        "Git",
        "GitHub"
    ],

     "TECHNOLOGY": [
        "React",
        "Node.js",
        "TensorFlow",
        "PyTorch"
    ],

    "DATABASE": [
        "MySQL",
        "PostgreSQL",
        "MongoDB",
        "Oracle"
    ],

    "CLOUD_PLATFORM": [
        "AWS",
        "Google Cloud",
        "Azure"
    ]
}

In [ ]:
matcher = PhraseMatcher(nlp.vocab, attr="LOWER")

In [ ]:
for entity_type, words in skill_dictionary.items():

    patterns = [nlp.make_doc(word)for word in words]

    matcher.add(entity_type, patterns)

In [ ]:
def extract_custom_entities(text):

    doc = nlp(str(text))

    matches = matcher(doc)

    entities = []

    for match_id, start, end in matches:

        entity_type = nlp.vocab.strings[match_id]

        entity_text = doc[start:end].text

        entities.append(f"{entity_text} ({entity_type})")

    return entities

In [ ]:
text = """
We are looking for a Data Scientist with Python,
Machine Learning, SQL and Power BI experience.
The candidate should also know AWS and PostgreSQL.
"""

result = extract_custom_entities(text)

print("Detected Entities:")

for entity in result:
    print(entity)

Detected Entities:
Python (SKILL)
Machine Learning (SKILL)
SQL (SKILL)
Power BI (TOOL)
AWS (CLOUD_PLATFORM)
PostgreSQL (DATABASE)


In [ ]:
df["custom_entities"] = df[text_column].apply(extract_custom_entities)

In [ ]:
print(df[[text_column, "custom_entities"]].head(10))

                                     job_description  \
0  We are looking for a Python Developer to join ...   
1  We are looking for a Data Scientist to join ou...   
2  We are looking for a Data Engineer to join our...   
3  We are looking for a Business Analyst to join ...   
4  We are looking for a Business Analyst to join ...   
5  We are looking for a Software Engineer to join...   
6  We are looking for a Frontend Developer to joi...   
7  We are looking for a AI Engineer to join our t...   
8  We are looking for a Frontend Developer to joi...   
9  We are looking for a AI Engineer to join our t...   

                                     custom_entities  
0      [Python (SKILL), Python (SKILL), SQL (SKILL)]  
1  [Python (SKILL), SQL (SKILL), Machine Learning...  
2                      [SQL (SKILL), Python (SKILL)]  
3                     [SQL (SKILL), Power BI (TOOL)]  
4                     [SQL (SKILL), Power BI (TOOL)]  
5  [Git (TOOL), C++ (SKILL), Java (SKILL), Python... 

In [ ]:
df["custom_entities"] = df["custom_entities"].apply(lambda x: ", ".join(x))

In [ ]:
df.to_csv("custom_ner_results.csv", index=False)

print("File saved successfully!")

File saved successfully!


In [ ]:
text_column = "job_description"